# DevOps LLM Fine-tuning with Unsloth
This notebook is configured to run on Google Colab's Free T4 GPU. It uses **Unsloth** to fine-tune a model like Llama 3.1 or Qwen 2.5 for DevOps-specific tasks (Docker, K8s, Terraform, CI/CD).

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit", # Options: Qwen2.5-7B, Mistral-v0.3
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### Load DevOps Dataset
We use ChatML format for the instructions.

In [ ]:
from datasets import load_dataset
dataset = load_dataset("m-a-p/CodeFeedback-Filtered-Instruction", split = "train")
# Filter for DevOps related keywords (manual step or pre-filtered dataset recommended)
# For this demo, we assume the dataset contains valid instruction/output pairs.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Increase for real training
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### Export to GGUF
After training, we export to GGUF to run on low-power devices/Ollama.

In [ ]:
model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")